# FARM quick start

Predict resistance for a single (drug SMILES, isolate) pair using a trained FARM checkpoint.

Before running:
1. Train your own checkpoint with `python -m farm.train ...` (see `farm/train.py --help`)
   **or** use the no-install web server at <https://cdc.biohpc.swmed.edu/farm/intro>.
2. Download the dataset from Zenodo: <https://doi.org/10.5281/zenodo.20217971>.
3. Fill in the placeholder paths below.

In [ ]:
# === Fill these in ===
CHECKPOINT_PATH = 'path/to/your/farm_checkpoint.h5'   # your trained FARM weights
DATA_ROOT       = 'path/to/FARM_dataset_v1'           # unpacked Zenodo record

# A drug + an isolate to score
DRUG_SMILES = 'CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O)O)C'  # ampicillin
ISOLATE_ID  = None        # set to a sample_id from the chosen cohort
COHORT      = 'antibiogram'

In [ ]:
import os, numpy as np, pandas as pd
from farm import predict, data as farm_data

In [ ]:
# Load the per-cohort VAMP features and pick one isolate.
vamp_path  = os.path.join(DATA_ROOT, 'ten_cohorts', f'{COHORT}_VAMP.txt')
vamp       = farm_data.load_vamp_features(vamp_path)

if ISOLATE_ID is None:
    ISOLATE_ID = next(iter(vamp))
    print(f'No ISOLATE_ID specified; using {ISOLATE_ID}.')

gene_presence, gene_mutation = vamp[ISOLATE_ID]
print(f'Gene-presence vector: {gene_presence.shape}, sum={gene_presence.sum()}')
print(f'Gene-mutation vector: {gene_mutation.shape}, sum={gene_mutation.sum()}')

In [ ]:
# Run inference.
result = predict.predict_one(
    smiles        = DRUG_SMILES,
    gene_presence = gene_presence,
    gene_mutation = gene_mutation,
    checkpoint    = CHECKPOINT_PATH,
    return_attention = True,
)

print(f'Predicted label      : {result["predicted_label"]}')
print(f'P(resistant)         : {result["probability_resistant"]:.4f}')

## Visualise the attention readouts

If your checkpoint exposes `atom_saliency` and `gene_attention` as model outputs, the next cell will plot them. See `notebooks/02_attention_visualization.ipynb` for the full Figure 3 / Figure 4 style visualisations.

In [ ]:
import matplotlib.pyplot as plt

if 'atom_saliency' in result:
    plt.figure(figsize=(8, 2))
    plt.bar(range(len(result['atom_saliency'])), result['atom_saliency'])
    plt.xlabel('Atom index'); plt.ylabel('Saliency')
    plt.title('Atom-level saliency for the input drug')
    plt.tight_layout(); plt.show()
else:
    print('Checkpoint did not expose atom_saliency; see notebook 02 for the manual extraction.')